# 04.03 — GQLAlchemy Backend

**Purpose:** Demonstrate that Orthograph models are fully compatible with
GQLAlchemy without a database connection. This notebook covers:

- Auto-generating GQLAlchemy `Node`/`Relationship` classes from Orthograph models
- Instantiating generated classes (Pydantic v1/v2 coexistence)
- Pre-save data validation using the Orthograph schema
- Cypher generation and static Cypher query validation
- Static query validation via `ValidatedQueryBuilder`

**No database connection is required.** Everything runs locally.

For database interaction (save, load, query execution), see
[03.04 -- GQLAlchemy Database Interaction](03.04_gqlalchemy_database_interaction.ipynb).

```bash
pip install orthograph[gqlalchemy,cypher]
```

## 1. Define the Schema

The schema is defined once using Orthograph's `NodeModel` and `RelationshipModel`.
This is the **single source of truth** for both validation and OGM.

In [ ]:
from shared.filmography import FILMOGRAPHY_MODEL, ActedIn, Movie, Person

from orthograph.api.model import GraphValidator
from orthograph.cypher.generator import CypherGenerator
from orthograph.diagnostics.result import GraphValidationError
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import Cardinality


graph_definition = FILMOGRAPHY_MODEL

## 2. Visualize the Schema

In [ ]:
from orthograph.api.visualization import render_model


print(render_model(graph_definition))

## 3. Auto-Generate GQLAlchemy Classes

`generate_gqlalchemy_classes()` translates Orthograph Pydantic v2 models into
GQLAlchemy Pydantic v1 `Node`/`Relationship` subclasses at runtime.
Both Pydantic versions coexist in the same process without conflict.

These generated classes are used internally by `GqlAlchemyClient` -- users
never need to import or interact with them directly.

In [ ]:
from orthograph.backends.gqlalchemy.codegen import generate_gqlalchemy_classes


schema = generate_gqlalchemy_classes(graph_definition)

print("Generated node classes:", list(schema.node_classes.keys()))
print("Generated rel classes: ", list(schema.rel_classes.keys()))

In [ ]:
PersonGqa = schema.get_node_class("Person")

print(f"Class name:   {PersonGqa.__name__}")
print(f"GQA label:    {PersonGqa.label}")
print(f"GQA labels:   {PersonGqa.labels}")
print(f"Annotations:  {PersonGqa.__annotations__}")

p = PersonGqa(name="Alice", born=1985)
print(f"\nInstance:     {p}")
print(f"Properties:   {p._properties}")

In [ ]:
ActedInGqa = schema.get_rel_class("ACTED_IN")

print(f"Class name:   {ActedInGqa.__name__}")
print(f"GQA type:     {ActedInGqa.type}")
print(f"Annotations:  {ActedInGqa.__annotations__}")

r = ActedInGqa(_start_node_id=0, _end_node_id=1, role="Neo")
print(f"\nInstance:     {r}")
print(f"Properties:   {r._properties}")

## 4. Pre-Save Validation

Orthograph's `GraphValidator` runs entirely in-process. Bad data is rejected
**before** any database call is attempted. This is the same validation that
`GqlAlchemyClient.save_node()` performs internally.

In [ ]:
validator = GraphValidator(graph_definition)

result = validator.validate_nodes(
    [{"__label__": "Person", "name": "Alice", "born": 1985}]
)
print(f"Valid data:   is_valid={result.is_valid}, errors={len(result.errors)}")

result = validator.validate_nodes([{"__label__": "Movie", "title": "Inception"}])
print(f"Missing year: is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes(
    [{"__label__": "Movie", "title": "Inception", "year": "twenty"}]
)
print(f"Wrong type:   is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes(
    [{"__label__": "Person", "name": "Alice", "born": 1985, "unknown": "x"}]
)
print(f"Extra props:  is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes([{"__label__": "City", "name": "NYC"}])
print(f"Unknown type: is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

## 5. Cypher Generation

In [ ]:
gen = CypherGenerator(graph_definition)

print("Uniqueness constraints:")
for constraint in gen.generate_constraints():
    print(f"  {constraint}")

query, params = gen.merge_node({"__label__": "Person", "name": "Alice", "born": 1985})
print(f"\nMERGE query:  {query}")
print(f"MERGE params: {params}")

query = gen.match_node(Person)
print(f"\nMATCH query:  {query}")

## 6. Static Cypher Query Validation

In [ ]:
from orthograph.cypher.parser import validate_cypher


result = validate_cypher(
    "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p.name, m.title",
    graph_definition,
)
print(f"Valid query:   {len(result.errors)} errors")

result = validate_cypher(
    "MATCH (s:Studio)-[:PRODUCED]->(m:Movie) RETURN s.name",
    graph_definition,
)
print(f"Invalid query: {len(result.errors)} errors")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")

## 7. ValidatedQueryBuilder -- Static Validation

`ValidatedQueryBuilder.validate_query()` validates Cypher against the schema
without executing anything. No database connection needed. Useful for CI/CD
checks or linting pipelines.

In [ ]:
from orthograph.backends.gqlalchemy.query_builder import ValidatedQueryBuilder


vqb = ValidatedQueryBuilder(graph_definition=graph_definition, db=None)

result = vqb.validate_query("MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p, m")
print(f"Valid:   is_valid={result.is_valid}")

result = vqb.validate_query("MATCH (s:Studio) RETURN s")
print(f"Invalid: is_valid={result.is_valid}")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")

result = vqb.validate_query("MATCH ()-[:PRODUCED]->() RETURN *")
print(f"Invalid: is_valid={result.is_valid}")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")

# 03.04 -- GQLAlchemy Database Interaction

**Purpose:** Demonstrate `GqlAlchemyClient` and `ValidatedQueryBuilder`
against a live graph database. The extension supports both **Neo4j** and
**Memgraph** through GQLAlchemy's vendor abstraction.

This notebook covers:

- Connecting to Neo4j or Memgraph (configurable switch)
- Saving nodes with pre-save schema validation
- Rejecting invalid data before it reaches the database
- Saving relationships with property validation
- Executing validated queries via `ValidatedQueryBuilder`
- Raw query execution (no validation)

For the offline features (codegen, static validation, Cypher generation),
see [03.03 -- GQLAlchemy Compatibility](03.03_gqlalchemy_integration.ipynb).

## Prerequisites

```bash
pip install orthograph[gqlalchemy,cypher]
```

**You need a running graph database.** Start one with Docker:

Neo4j:
```bash
docker run -d --name neo4j -p 7687:7687 -p 7474:7474 \
  -e NEO4J_AUTH=neo4j/test neo4j:5
```

Memgraph:
```bash
docker run -d --name memgraph -p 7687:7687 memgraph/memgraph
```

See [Neo4j Docker docs](https://neo4j.com/docs/operations-manual/current/docker/)
or [Memgraph Quick Start](https://memgraph.com/docs/getting-started) for details.

## 1. Configuration

Set `DATABASE` to `"neo4j"` or `"memgraph"` and adjust the connection
parameters to match your setup.

In [ ]:
from shared.utils import load_env


# Load credentials from .env file (or .env_default for defaults)
DATABASE = load_env("DATABASE", "neo4j")  # "neo4j" or "memgraph"

NEO4J_HOST = (
    load_env("NEO4J_URI", "bolt://localhost:7687").replace("bolt://", "").split(":")[0]
)
NEO4J_PORT = int(load_env("NEO4J_URI", "bolt://localhost:7687").split(":")[-1])
NEO4J_USER = load_env("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = load_env("NEO4J_PASSWORD", "password")

MEMGRAPH_HOST = (
    load_env("MEMGRAPH_URI", "bolt://localhost:7688")
    .replace("bolt://", "")
    .split(":")[0]
)
MEMGRAPH_PORT = int(load_env("MEMGRAPH_URI", "bolt://localhost:7688").split(":")[-1])

## 2. Define the Schema

Same model as in 03.03 -- the single source of truth for validation and OGM.

In [ ]:
from typing import Optional

from orthograph.graph_definition.models import NodeModel, RelationshipModel


class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):  # noqa: F811
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int
    tagline: Optional[str] = None


class ActedIn(RelationshipModel):  # noqa: F811
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    __source_cardinality__ = Cardinality.ZERO_OR_MORE
    __target_cardinality__ = Cardinality.ONE_OR_MORE
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_label__ = "Person"
    __target_label__ = "Movie"


graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn, Directed],
)

print(f"Model: {graph_definition.name}")
print(f"Node types: {graph_definition.node_labels}")
print(f"Relationship types: {graph_definition.relationship_labels}")

## 3. Connect to the Database

In [ ]:
from orthograph.backends.gqlalchemy.client import GqlAlchemyClient


if DATABASE == "neo4j":
    from gqlalchemy import Neo4j

    db = Neo4j(
        host=NEO4J_HOST,
        port=NEO4J_PORT,
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
    )
elif DATABASE == "memgraph":
    from gqlalchemy import Memgraph

    db = Memgraph(host=MEMGRAPH_HOST, port=MEMGRAPH_PORT)
else:
    raise ValueError(f"Unknown DATABASE: {DATABASE!r}. Use 'neo4j' or 'memgraph'.")

client = GqlAlchemyClient(graph_definition=graph_definition, db=db, backend=DATABASE)
print(
    f"Connected to {DATABASE}. Schema has {len(client.schema.node_classes)} node classes."
)

## 4. Clean Slate

Delete all existing data so the notebook is idempotent.

In [ ]:
client.execute("MATCH (n) DETACH DELETE n")
print("Database cleared.")

## 5. Save Nodes with Validation

The client validates the data dict against the Orthograph schema
**before** writing to the database. If validation fails, a
`GraphValidationError` is raised and no write is attempted.

In [ ]:
alice = client.save_node({"name": "Alice", "born": 1985}, node_type="Person")
bob = client.save_node({"name": "Bob"}, node_type="Person")
matrix = client.save_node({"title": "The Matrix", "year": 1999}, node_type="Movie")
inception = client.save_node(
    {
        "title": "Inception",
        "year": 2010,
        "tagline": "Your mind is the scene of the crime.",
    },
    node_type="Movie",
)

print(f"Saved Person: name={alice.name}")
print(f"Saved Person: name={bob.name}")
print(f"Saved Movie:  title={matrix.title}, year={matrix.year}")
print(f"Saved Movie:  title={inception.title}, year={inception.year}")

## 6. Validation Rejects Bad Data

Invalid data is caught by Orthograph before any database call.

In [ ]:
# Missing required field: 'year' is required on Movie
try:
    client.save_node({"title": "Bad Movie"}, node_type="Movie")
except GraphValidationError as e:
    print(f"Rejected (missing year):\n{e}\n")

# Wrong type: 'year' should be int, not str
try:
    client.save_node({"title": "Bad Movie", "year": "twenty"}, node_type="Movie")
except GraphValidationError as e:
    print(f"Rejected (wrong type):\n{e}\n")

# Unknown node type: 'City' is not in the model
try:
    client.save_node({"name": "NYC"}, node_type="City")
except GraphValidationError as e:
    print(f"Rejected (unknown type):\n{e}")

## 7. Save Relationships with Validation

In [ ]:
client.save_relationship(
    {"role": "Neo"},
    rel_type="ACTED_IN",
    start_node_id=alice._id,
    end_node_id=matrix._id,
)
print("Saved ACTED_IN: Alice -> The Matrix")

client.save_relationship(
    {},
    rel_type="DIRECTED",
    start_node_id=alice._id,
    end_node_id=inception._id,
)
print("Saved DIRECTED: Alice -> Inception")

# Invalid: missing required 'role' property on ACTED_IN
try:
    client.save_relationship(
        {},
        rel_type="ACTED_IN",
        start_node_id=bob._id,
        end_node_id=matrix._id,
    )
except GraphValidationError as e:
    print(f"\nRejected (missing role):\n{e}")

## 8. Validated Query Execution

`ValidatedQueryBuilder` validates the Cypher against the schema before
executing it. Invalid queries are rejected without touching the database.

In [ ]:
from orthograph.backends.gqlalchemy.query_builder import ValidatedQueryBuilder


vqb = ValidatedQueryBuilder(graph_definition=graph_definition, db=db)

# Valid query -- passes validation and executes
results = vqb.execute_validated(
    "MATCH (p:Person) RETURN p.name AS name, p.born AS born"
)
print("Persons in the database:")
for row in results:
    print(f"  {row}")

print()

# Invalid query -- rejected before execution
try:
    vqb.execute_validated("MATCH (s:Studio) RETURN s")
except GraphValidationError as e:
    print(f"Rejected (unknown label):\n{e}")

## 9. Raw Query Execution

For queries that do not need schema validation, `client.execute()` is a
direct passthrough to the database.

In [ ]:
results = client.execute("MATCH (n) RETURN labels(n) AS labels, count(n) AS count")
print("Node counts by label:")
for row in results:
    print(f"  {row}")

## 10. Cleanup

In [ ]:
client.execute("MATCH (n) DETACH DELETE n")
print("Database cleaned up.")